# Interactive Skeleton Builder for URDF

Data preprocessing step
--> needs manual intervention of selecting keypoints

This notebook helps you:
1. Load and visualize your URDF mesh (fully interactive 3D)
2. Explore coordinates by hovering and clicking
3. Define keypoints and joints
4. Create skeleton lines between points
5. Export your skeleton definition

**Instructions:**
- Run cells in order
- **Rotate, zoom, pan** the 3D view with your mouse
- **Hover** over points to see coordinates
- Click points to select them

In [ ]:
# Imports
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial.transform import Rotation as R
import xml.etree.ElementTree as ET
import os

print("✓ Imports loaded")
print("\nTip: Graphs are fully interactive!")
print("  - Left click + drag: Rotate")
print("  - Right click + drag: Pan")
print("  - Scroll: Zoom")
print("  - Hover: See coordinates")

## 1. Load URDF and Meshes

In [ ]:
# Configuration
urdf_path = "/home/caee/Desktop/Excavator_BC_RL/excavatorURDF/excavator_lowpoly_locked_splitbucket.urdf"
urdf_dir = os.path.dirname(urdf_path)

# STL loader
def load_stl(filepath):
    """Load binary STL file, returns Nx3x3 array of triangles"""
    with open(filepath, 'rb') as f:
        f.read(80)  # Skip header
        n_triangles = np.frombuffer(f.read(4), dtype=np.uint32)[0]
        vertices = []
        for _ in range(n_triangles):
            f.read(12)  # Skip normal
            v1 = np.frombuffer(f.read(12), dtype=np.float32)
            v2 = np.frombuffer(f.read(12), dtype=np.float32)
            v3 = np.frombuffer(f.read(12), dtype=np.float32)
            vertices.append([v1.copy(), v2.copy(), v3.copy()])
            f.read(2)  # Skip attribute
        return np.array(vertices)

def make_transform(xyz, rpy):
    """Create 4x4 homogeneous transform from xyz + rpy"""
    T = np.eye(4)
    T[:3, :3] = R.from_euler('xyz', rpy).as_matrix()
    T[:3, 3] = xyz
    return T

# Parse URDF
print('Loading URDF and building kinematic chain...')
tree = ET.parse(urdf_path)
root = tree.getroot()

# --- Build kinematic tree ---
joints = {}
child_to_joint = {}
parent_links = set()
child_links = set()

for joint in root.findall('joint'):
    jname = joint.get('name')
    jtype = joint.get('type')
    parent = joint.find('parent').get('link')
    child = joint.find('child').get('link')
    origin = joint.find('origin')
    if origin is not None:
        xyz = [float(x) for x in origin.get('xyz', '0 0 0').split()]
        rpy = [float(x) for x in origin.get('rpy', '0 0 0').split()]
    else:
        xyz = [0, 0, 0]
        rpy = [0, 0, 0]
    joints[jname] = {'parent': parent, 'child': child, 'xyz': xyz, 'rpy': rpy, 'type': jtype}
    child_to_joint[child] = joints[jname]
    parent_links.add(parent)
    child_links.add(child)

# Find root link (parent but never a child)
root_link = (parent_links - child_links).pop()
print(f'  Root link: {root_link}')

# Build world transforms for every link via BFS
link_world_transforms = {root_link: np.eye(4)}

parent_to_children = {}
for jname, jinfo in joints.items():
    p = jinfo['parent']
    if p not in parent_to_children:
        parent_to_children[p] = []
    parent_to_children[p].append(jinfo)

queue = [root_link]
while queue:
    current = queue.pop(0)
    if current in parent_to_children:
        for jinfo in parent_to_children[current]:
            child = jinfo['child']
            joint_T = make_transform(jinfo['xyz'], jinfo['rpy'])
            link_world_transforms[child] = link_world_transforms[current] @ joint_T
            queue.append(child)

print(f'  Computed world transforms for {len(link_world_transforms)} links')

# --- Load ALL visual meshes (some links have multiple visuals) ---
visual_elements = []

for link in root.findall('link'):
    link_name = link.get('name')
    link_T = link_world_transforms.get(link_name, np.eye(4))

    for visual in link.findall('visual'):  # findall to get ALL visuals
        geometry = visual.find('geometry')
        if geometry is None:
            continue
        mesh = geometry.find('mesh')
        if mesh is None:
            continue

        mesh_file = mesh.get('filename')
        mesh_path = os.path.join(urdf_dir, mesh_file)
        if not os.path.exists(mesh_path):
            continue

        origin = visual.find('origin')
        if origin is not None:
            v_xyz = [float(x) for x in origin.get('xyz', '0 0 0').split()]
            v_rpy = [float(x) for x in origin.get('rpy', '0 0 0').split()]
        else:
            v_xyz = [0, 0, 0]
            v_rpy = [0, 0, 0]

        visual_T = make_transform(v_xyz, v_rpy)
        world_T = link_T @ visual_T

        triangles = load_stl(mesh_path)
        visual_elements.append({
            'link_name': link_name,
            'mesh_file': mesh_file,
            'triangles': triangles,
            'world_transform': world_T,
            'link_transform': link_T,
        })
        print(f'  {link_name}/{mesh_file}: {triangles.shape[0]} triangles')

print(f'\nLoaded {len(visual_elements)} visual elements')
print('All links positioned via full kinematic chain.')

## 2. Coordinate Helpers + Full Excavator View

Helper functions to convert between world and local coordinates, then render the full excavator with proper triangle meshes.

In [ ]:
# --- Coordinate conversion helpers ---
def world_to_local(world_xyz, link_name):
    """Convert world coordinates to local offset for a given link.
    Use this to input keypoints: hover over the mesh to get world coords,
    then call world_to_local([x, y, z], 'link_name') to get the offset."""
    T = link_world_transforms[link_name]
    world_pt = np.array([*world_xyz, 1.0])
    local_pt = np.linalg.inv(T) @ world_pt
    return local_pt[:3]

def local_to_world(local_xyz, link_name):
    """Convert local offset back to world coordinates."""
    T = link_world_transforms[link_name]
    local_pt = np.array([*local_xyz, 1.0])
    world_pt = T @ local_pt
    return world_pt[:3]

# --- Helper to transform and add a mesh as Mesh3d ---
def add_mesh_trace(fig, elem, color='gray', opacity=0.8, show_hover=True):
    """Add a visual element as a proper Mesh3d triangle mesh to a figure."""
    triangles = elem['triangles']
    world_T = elem['world_transform']
    link_name = elem['link_name']

    # Transform all vertices to world frame
    all_verts = []
    for tri in triangles:
        for v in tri:
            v_world = world_T @ np.array([*v, 1.0])
            all_verts.append(v_world[:3])
    all_verts = np.array(all_verts)

    n_tri = len(triangles)
    i_idx = [t * 3 for t in range(n_tri)]
    j_idx = [t * 3 + 1 for t in range(n_tri)]
    k_idx = [t * 3 + 2 for t in range(n_tri)]

    hover = (
        f'<b>{link_name}</b><br>'
        'X: %{x:.4f}<br>'
        'Y: %{y:.4f}<br>'
        'Z: %{z:.4f}<br>'
        '<extra></extra>'
    ) if show_hover else None

    fig.add_trace(go.Mesh3d(
        x=all_verts[:, 0], y=all_verts[:, 1], z=all_verts[:, 2],
        i=i_idx, j=j_idx, k=k_idx,
        color=color, opacity=opacity,
        name=link_name,
        hovertemplate=hover,
        hoverinfo='skip' if not show_hover else None,
        showlegend=show_hover,
        lighting=dict(ambient=0.7, diffuse=0.8, specular=0.3),
        flatshading=True,
    ))

# --- Render full excavator ---
fig = go.Figure()

body_colors = {
    'compact_excavator_cabin_body_cmpl': 'steelblue',
    'compact_excavator_turret_cabin_roller': 'slategray',
    'compact_excavator_frame_body': 'cadetblue',
    'compact_excavator_main_drive_wheel_cmpl_(1)': 'dimgray',
    'compact_excavator_main_drive_wheel_cmpl': 'dimgray',
    'compact_excavator_front_wheel': 'dimgray',
    'compact_excavator_front_wheel_2': 'dimgray',
    'track_roller_wheel_1': 'gray', 'track_roller_wheel_1_2': 'gray',
    'track_roller_wheel_1_3': 'gray', 'track_roller_wheel_1_4': 'gray',
    'track_roller_wheel_1_5': 'gray', 'track_roller_wheel_1_6': 'gray',
    'carrier_roller': 'silver', 'carrier_roller_2': 'silver',
    'part01_pin_1': 'red', 'part02_cmpl': 'orange',
    'part03': 'yellow', 'part04': 'green',
}

for elem in visual_elements:
    color = body_colors.get(elem['link_name'], 'gray')
    add_mesh_trace(fig, elem, color=color, opacity=0.8, show_hover=True)

fig.update_layout(
    title='Full Excavator (Mesh3d) - Hover for world coordinates on any part',
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data',
        camera=dict(eye=dict(x=2.0, y=2.0, z=1.5))
    ),
    width=1400, height=900, hovermode='closest'
)

print('Full excavator view ready!')
print('  Hover over any part to see WORLD coordinates.')
print('  Use world_to_local([x,y,z], link_name) to convert for keypoints.')
fig.show()

## 3. Arm-Only View (Color-Coded)

Focused view of just the arm parts for detailed keypoint placement:

In [ ]:
# Arm-only view with Mesh3d
fig_arm = go.Figure()

arm_colors = {
    'part01_pin_1': 'red', 'upper_boom': 'orange',
    'lower_boom': 'yellow', 'bucketry': 'green',
}

for elem in visual_elements:
    if elem['link_name'] in arm_colors:
        add_mesh_trace(fig_arm, elem, color=arm_colors[elem['link_name']],
                       opacity=0.8, show_hover=True)

fig_arm.update_layout(
    title='Arm Only (Mesh3d) - Hover for world coordinates',
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data',
        camera=dict(eye=dict(x=2.0, y=2.0, z=1.5))
    ),
    width=1400, height=900, hovermode='closest'
)

print('Arm view ready!')
print('  Red=part01 (base), Orange=part02 (boom), Yellow=part03 (stick), Green=part04 (bucket)')
fig_arm.show()

## 4. Define Keypoints (World Coordinates)

Define keypoints using the **world coordinates** you see when hovering over the mesh.  
`world_to_local()` automatically converts them to local offsets for each link.

In [ ]:
# Define keypoints using WORLD coordinates (what you see when hovering)
# world_to_local() converts them to the local offset for the link
# Order matches KEYPOINT_ORDER: 0:bucket_tip 1:stick_tip 2:bucket_floor 3:boom_tip
#                                4:arm_base   5:turret_center 6:frame_front_mid 7:frame_rear_mid

keypoints_list = [
    ("bucket_tip",      "bucketry",
     world_to_local([0.000, -4.500, 0.500], 'bucketry')),
    ("stick_tip",       "lower_boom",
     world_to_local([0.000, -4.000, 1.400], 'lower_boom')),
    ("bucket_floor",    "bucketry",
     world_to_local([0.000, -4.650, 1.125], 'bucketry')),
    ("boom_tip",        "upper_boom",
     world_to_local([0.000, -2.800, 1.800], 'upper_boom')),
    ("arm_base",        "part01_pin_1",
     world_to_local([0.000, -0.900, 0.700], 'part01_pin_1')),
    ("turret_center",   "compact_excavator_turret_cabin_roller",
     world_to_local([0.000, 0.000, 0.000], 'compact_excavator_turret_cabin_roller')),
    ("frame_front_mid", "compact_excavator_frame_body",
     world_to_local([0.000, -0.800, 0.000], 'compact_excavator_frame_body')),
    ("frame_rear_mid",  "compact_excavator_frame_body",
     world_to_local([0.000,  0.800, 0.000], 'compact_excavator_frame_body')),
]

# Convert to dictionary
keypoints = {name: (link, offset) for name, link, offset in keypoints_list}

print(f'Defined {len(keypoints_list)} keypoints:')
print('='*70)
for i, (name, link, offset) in enumerate(keypoints_list):
    world_pos = local_to_world(offset, link)
    print(f'  {i}: {name:20s} on {link}')
    print(f'    World: [{world_pos[0]:.4f}, {world_pos[1]:.4f}, {world_pos[2]:.4f}]')
    print(f'    Local: [{offset[0]:.6f}, {offset[1]:.6f}, {offset[2]:.6f}]')
print()
print('To adjust: hover the mesh, copy world coords, replace the [x,y,z] above.')

## 5. Define Skeleton Lines (Full Body)

Connect keypoints to form the full body skeleton:

In [ ]:
# Define arm joints (in kinematic chain order)
arm_joints = ["lower_arm", "upperToLow", "scoop1"]

# Define skeleton connections (full body)
skeleton_lines = [
    # Body
    ("frame_front_mid", "turret_center"),
    ("turret_center", "frame_rear_mid"),
    # Turret to arm
    ("turret_center", "arm_base"),
    # Arm chain
    ("arm_base", "boom_tip"),
    ("boom_tip", "stick_tip"),
    ("stick_tip", "bucket_floor"),
    ("bucket_floor", "bucket_tip"),
    ("bucket_tip", "stick_tip"),  # Close bucket triangle
]

print(f'Arm joints ({len(arm_joints)}): {arm_joints}')
print(f'Skeleton connections ({len(skeleton_lines)} lines):')
for p1, p2 in skeleton_lines:
    print(f'  {p1} -> {p2}')

## 6. Visualize Complete Skeleton (Full Body)

See your skeleton overlaid on the **full** excavator mesh!

In [ ]:
# Skeleton overlaid on full body (Mesh3d)
fig_complete = go.Figure()

# Add ALL meshes semi-transparent
for elem in visual_elements:
    add_mesh_trace(fig_complete, elem, color='lightgray', opacity=0.3, show_hover=False)

# Compute keypoint world positions
keypoint_positions = {}
for name, (link, offset) in keypoints.items():
    keypoint_positions[name] = local_to_world(offset, link)

# Plot keypoints
if keypoint_positions:
    kp_names = list(keypoint_positions.keys())
    kp_coords = np.array([keypoint_positions[n] for n in kp_names])

    fig_complete.add_trace(go.Scatter3d(
        x=kp_coords[:, 0], y=kp_coords[:, 1], z=kp_coords[:, 2],
        mode='markers+text',
        marker=dict(size=10, color='red', symbol='circle',
                    line=dict(width=2, color='black')),
        text=kp_names,
        textposition='top center',
        textfont=dict(size=10, color='darkred'),
        name='Keypoints',
        hovertemplate='<b>%{text}</b><br>'
                      'X: %{x:.4f}<br>'
                      'Y: %{y:.4f}<br>'
                      'Z: %{z:.4f}<br>'
                      '<extra></extra>'
    ))

    # Draw skeleton lines
    for p1_name, p2_name in skeleton_lines:
        if p1_name in keypoint_positions and p2_name in keypoint_positions:
            p1 = keypoint_positions[p1_name]
            p2 = keypoint_positions[p2_name]
            fig_complete.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='blue', width=8),
                name=f'{p1_name}-{p2_name}',
                showlegend=False, hoverinfo='skip'
            ))

fig_complete.update_layout(
    title='Complete Skeleton (Full Body)',
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data',
        camera=dict(eye=dict(x=2.0, y=2.0, z=1.5))
    ),
    width=1400, height=900
)

print('Skeleton visualization complete!')
print('  Red dots = Keypoints')
print('  Blue lines = Skeleton connections')
fig_complete.show()

## 7. Export Skeleton Definition

Copy this code to use in your scripts:

In [ ]:
print('='*70)
print('COPY THIS CODE TO USE YOUR SKELETON')
print('='*70)
print()
print('from urdf_skeleton_custom import build_excavator_skeleton')
print('import numpy as np')
print()
print('# Update build_excavator_skeleton() in urdf_skeleton_custom.py with:')
print(f'urdf_path = "{urdf_path}"')
print(f'arm_joints = {arm_joints}')
print('keypoints = [')
for name, link, offset in keypoints_list:
    print(f'    ("{name}", "{link}", np.array([{offset[0]:.6f}, {offset[1]:.6f}, {offset[2]:.6f}])),')
print(']')
print()
print('# --- OR use directly without build_excavator_skeleton: ---')
print('from urdf_skeleton_custom import CustomURDFSkeleton')
print(f'skeleton = CustomURDFSkeleton("{urdf_path}")')
print(f'skeleton.define_skeleton({arm_joints}, keypoints)')
print()
print('# Compute forward kinematics')
print(f'joint_angles = np.zeros({len(arm_joints)})  # {arm_joints}')
print('keypoints_3d = skeleton.forward_kinematics(joint_angles)')
print()
print('='*70)
print()
print('Skeleton connections (for visualization):')
print('skeleton_lines = [')
for p1, p2 in skeleton_lines:
    print(f'    ("{p1}", "{p2}"),')
print(']')
print()
print('='*70)
print(f'Total: {len(keypoints_list)} keypoints (body + arm)')
print('All coordinates automatically converted from world to local.')